# 📊 Proyecto de Análisis de Datos: Valores UF 2025

**Objetivo:** Extraer, limpiar y analizar los valores de la Unidad de Fomento (UF) del año 2025 desde el sitio web del SII.

**Autor:** Analista de Datos  
**Fecha:** Febrero 2025  
**Fuente:** https://www.sii.cl/valores_y_fechas/uf/uf2025.htm

## 📚 1. Importación de Librerías

In [ ]:
# Importar las librerías necesarias
import pandas as pd
import numpy as np
import warnings
from datetime import datetime

# Ignorar advertencias para una salida más limpia
warnings.filterwarnings('ignore')

# Configuración de pandas para mejor visualización
pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', 100)
pd.set_option('display.float_format', '{:.2f}'.format)

print("✅ Librerías importadas correctamente")
print(f"Versión Pandas: {pd.__version__}")
print(f"Versión NumPy: {np.__version__}")

## 🌐 2. Extracción de Datos desde Web (read_html)

In [ ]:
# URL del sitio web del SII con los valores UF
url = 'https://www.sii.cl/valores_y_fechas/uf/uf2025.htm'

# Extraer todas las tablas HTML de la página
print("🔄 Extrayendo datos desde el sitio web del SII...")
try:
    # read_html devuelve una lista de DataFrames (uno por cada tabla encontrada)
    tablas = pd.read_html(url, decimal=',', thousands='.')
    print(f"✅ Se encontraron {len(tablas)} tabla(s) en la página")
    
    # La primera tabla contiene los valores UF
    df_web = tablas[0]
    print(f"✅ Tabla extraída con dimensiones: {df_web.shape}")
    print("\n👁️ Primeras filas del DataFrame extraído:")
    display(df_web.head())
    
except Exception as e:
    print(f"❌ Error al extraer datos: {e}")
    print("⚠️ Usando archivo CSV local como alternativa...")
    df_web = pd.read_csv('UF_2025.csv', sep=';', decimal=',', thousands='.')

## 📂 3. Carga de Datos desde Archivo CSV Local

In [ ]:
# Leer el archivo CSV local
print("📥 Cargando datos desde archivo CSV local...")
df_csv = pd.read_csv('UF_2025.csv', sep=';', encoding='utf-8-sig')

print(f"✅ Datos cargados correctamente")
print(f"Dimensiones: {df_csv.shape[0]} filas x {df_csv.shape[1]} columnas")
print("\n👁️ Primeras 5 filas:")
display(df_csv.head())

print("\n📋 Información del DataFrame:")
print(df_csv.info())

## 🔧 4. Limpieza Inicial de Datos

In [ ]:
# Crear una copia para trabajar
df = df_csv.copy()

print("🔍 Estado inicial del DataFrame:")
print(f"Columnas: {df.columns.tolist()}")
print(f"\nTipos de datos:\n{df.dtypes}")
print(f"\nValores nulos por columna:\n{df.isnull().sum()}")

## 🧹 5. Ensuciamiento Intencional de Datos

Para simular un escenario real, vamos a introducir diferentes tipos de problemas comunes en los datos:

In [ ]:
# Establecer semilla para reproducibilidad
np.random.seed(42)

print("🎭 Ensuciando datos para simular escenario real...\n")

# 1. Introducir valores nulos aleatorios (5% de los datos)
print("1️⃣ Introduciendo valores nulos aleatorios...")
meses = ['Ene', 'Feb', 'Mar', 'Abr', 'May', 'Jun', 'Jul', 'Ago', 'Sep', 'Oct', 'Nov', 'Dic']
for mes in meses:
    if mes in df.columns:
        indices_nulos = np.random.choice(df.index, size=int(len(df) * 0.05), replace=False)
        df.loc[indices_nulos, mes] = np.nan

# 2. Introducir duplicados (agregar filas duplicadas)
print("2️⃣ Introduciendo filas duplicadas...")
filas_duplicar = df.sample(n=3, random_state=42)
df = pd.concat([df, filas_duplicar], ignore_index=True)

# 3. Introducir valores atípicos (outliers)
print("3️⃣ Introduciendo valores atípicos (outliers)...")
for mes in ['Ene', 'Feb', 'Mar']:
    if mes in df.columns:
        idx = np.random.choice(df.index, 2)
        df.loc[idx, mes] = '99.999,99'  # Valor extremadamente alto

# 4. Introducir espacios en blanco
print("4️⃣ Introduciendo espacios en blanco...")
for mes in ['Abr', 'May']:
    if mes in df.columns:
        idx = np.random.choice(df.index, 2)
        df.loc[idx, mes] = df.loc[idx, mes].astype(str) + '  '  # Espacios al final

# 5. Introducir formatos inconsistentes
print("5️⃣ Introduciendo formatos inconsistentes...")
df.loc[5, 'Día'] = '6.'  # Agregar punto al número de día
df.loc[10, 'Día'] = 'Día 11'  # Formato de texto

# 6. Crear columna con datos irrelevantes
print("6️⃣ Agregando columna con datos irrelevantes...")
df['Comentarios'] = np.random.choice(['OK', 'Revisar', '', np.nan], size=len(df))

print("\n✅ Datos ensuciados correctamente")
print(f"\nNuevas dimensiones: {df.shape}")
print(f"Valores nulos totales: {df.isnull().sum().sum()}")
print(f"\n👁️ Muestra de datos ensuciados:")
display(df.head(15))

## 🧼 6. Proceso de Limpieza de Datos

Ahora vamos a limpiar todos los problemas introducidos:

In [ ]:
print("🧼 Iniciando proceso de limpieza de datos...\n")

# PASO 1: Eliminar columnas irrelevantes
print("📌 PASO 1: Eliminando columnas irrelevantes...")
columnas_antes = df.columns.tolist()
if 'Comentarios' in df.columns:
    df = df.drop('Comentarios', axis=1)
print(f"   Columnas eliminadas: {[col for col in columnas_antes if col not in df.columns]}")

# PASO 2: Eliminar filas duplicadas
print("\n📌 PASO 2: Eliminando filas duplicadas...")
filas_antes = len(df)
df = df.drop_duplicates()
df = df.reset_index(drop=True)
print(f"   Filas duplicadas eliminadas: {filas_antes - len(df)}")

# PASO 3: Limpiar columna 'Día'
print("\n📌 PASO 3: Limpiando columna 'Día'...")
df['Día'] = df['Día'].astype(str).str.replace('.', '', regex=False)
df['Día'] = df['Día'].str.extract(r'(\d+)')[0]
df['Día'] = pd.to_numeric(df['Día'], errors='coerce')
print(f"   ✅ Columna 'Día' convertida a numérico")

# PASO 4: Limpiar y convertir valores UF
print("\n📌 PASO 4: Limpiando y convirtiendo valores UF...")
meses = ['Ene', 'Feb', 'Mar', 'Abr', 'May', 'Jun', 'Jul', 'Ago', 'Sep', 'Oct', 'Nov', 'Dic']

def limpiar_valor_uf(valor):
    """Función para limpiar y convertir valores UF a float"""
    if pd.isna(valor):
        return np.nan
    
    # Convertir a string y limpiar espacios
    valor_str = str(valor).strip()
    
    # Reemplazar separadores
    valor_str = valor_str.replace('.', '')  # Eliminar separador de miles
    valor_str = valor_str.replace(',', '.')  # Cambiar separador decimal
    
    try:
        return float(valor_str)
    except:
        return np.nan

for mes in meses:
    if mes in df.columns:
        df[mes] = df[mes].apply(limpiar_valor_uf)

print(f"   ✅ Valores UF convertidos a formato numérico")

# PASO 5: Eliminar outliers extremos
print("\n📌 PASO 5: Identificando y eliminando outliers...")
outliers_eliminados = 0
for mes in meses:
    if mes in df.columns:
        Q1 = df[mes].quantile(0.25)
        Q3 = df[mes].quantile(0.75)
        IQR = Q3 - Q1
        limite_inferior = Q1 - 3 * IQR
        limite_superior = Q3 + 3 * IQR
        
        # Contar outliers
        outliers_count = ((df[mes] < limite_inferior) | (df[mes] > limite_superior)).sum()
        outliers_eliminados += outliers_count
        
        # Reemplazar outliers con NaN
        df.loc[(df[mes] < limite_inferior) | (df[mes] > limite_superior), mes] = np.nan

print(f"   ⚠️ Outliers identificados y marcados como NaN: {outliers_eliminados}")

# PASO 6: Manejo de valores nulos
print("\n📌 PASO 6: Manejo de valores nulos...")
nulos_antes = df.isnull().sum().sum()
print(f"   Valores nulos encontrados: {nulos_antes}")

# Opción 1: Imputación con interpolación lineal (mejor para datos temporales)
for mes in meses:
    if mes in df.columns:
        df[mes] = df[mes].interpolate(method='linear', limit_direction='both')

nulos_despues = df.isnull().sum().sum()
print(f"   ✅ Valores nulos después de interpolación: {nulos_despues}")

# PASO 7: Ordenar por día
print("\n📌 PASO 7: Ordenando datos por día...")
df = df.sort_values('Día').reset_index(drop=True)
print(f"   ✅ Datos ordenados correctamente")

print("\n" + "="*60)
print("✅ LIMPIEZA COMPLETADA EXITOSAMENTE")
print("="*60)
print(f"\nDimensiones finales: {df.shape}")
print(f"Valores nulos restantes: {df.isnull().sum().sum()}")
print("\n👁️ Vista de datos limpios:")
display(df.head(10))

## 📊 7. Análisis Descriptivo de los Datos

In [ ]:
print("📊 ANÁLISIS ESTADÍSTICO DESCRIPTIVO\n")
print("="*60)

# 7.1 Información general del DataFrame
print("\n📋 1. INFORMACIÓN GENERAL DEL DATAFRAME")
print("-"*60)
print(df.info())

# 7.2 Estadísticas descriptivas básicas
print("\n📈 2. ESTADÍSTICAS DESCRIPTIVAS BÁSICAS")
print("-"*60)
display(df.describe())

# 7.3 Estadísticas adicionales
print("\n📊 3. ESTADÍSTICAS ADICIONALES POR MES")
print("-"*60)

estadisticas_mes = pd.DataFrame({
    'Promedio': df[meses].mean(),
    'Mediana': df[meses].median(),
    'Desv_Std': df[meses].std(),
    'Mínimo': df[meses].min(),
    'Máximo': df[meses].max(),
    'Rango': df[meses].max() - df[meses].min(),
    'Coef_Variación_%': (df[meses].std() / df[meses].mean() * 100).round(2)
})

display(estadisticas_mes)

# 7.4 Valores extremos
print("\n⚡ 4. VALORES EXTREMOS (TOP 5)")
print("-"*60)
print("\n🔝 Valores UF más altos por mes:")
for mes in meses:
    if mes in df.columns:
        max_val = df[mes].max()
        dia_max = df.loc[df[mes] == max_val, 'Día'].values[0]
        print(f"   {mes}: ${max_val:,.2f} (Día {dia_max})")

print("\n🔽 Valores UF más bajos por mes:")
for mes in meses:
    if mes in df.columns:
        min_val = df[mes].min()
        dia_min = df.loc[df[mes] == min_val, 'Día'].values[0]
        print(f"   {mes}: ${min_val:,.2f} (Día {dia_min})")

## 📈 8. Análisis de Tendencias y Correlaciones

In [ ]:
print("📈 ANÁLISIS DE TENDENCIAS\n")
print("="*60)

# 8.1 Variación mensual
print("\n📊 1. VARIACIÓN PROMEDIO MENSUAL")
print("-"*60)

variacion_mensual = pd.DataFrame()
for i in range(len(meses)-1):
    mes_actual = meses[i]
    mes_siguiente = meses[i+1]
    if mes_actual in df.columns and mes_siguiente in df.columns:
        prom_actual = df[mes_actual].mean()
        prom_siguiente = df[mes_siguiente].mean()
        variacion = prom_siguiente - prom_actual
        variacion_pct = (variacion / prom_actual) * 100
        
        print(f"{mes_actual} → {mes_siguiente}: ${variacion:+.2f} ({variacion_pct:+.3f}%)")

# 8.2 Matriz de correlación
print("\n🔗 2. MATRIZ DE CORRELACIÓN ENTRE MESES")
print("-"*60)
correlacion = df[meses].corr()
display(correlacion.round(3))

# 8.3 Resumen del año
print("\n📅 3. RESUMEN ANUAL 2025")
print("-"*60)
uf_inicio = df['Ene'].iloc[0]
uf_fin = df['Dic'].iloc[-1]
variacion_anual = uf_fin - uf_inicio
variacion_anual_pct = (variacion_anual / uf_inicio) * 100

print(f"UF al inicio del año (01-Ene): ${uf_inicio:,.2f}")
print(f"UF al final del año (31-Dic): ${uf_fin:,.2f}")
print(f"Variación anual: ${variacion_anual:+,.2f} ({variacion_anual_pct:+.2f}%)")
print(f"\nUF promedio del año: ${df[meses].mean().mean():,.2f}")
print(f"UF mínima del año: ${df[meses].min().min():,.2f}")
print(f"UF máxima del año: ${df[meses].max().max():,.2f}")

## 💾 9. Exportación de Datos Limpios

In [ ]:
print("💾 Exportando datos limpios...\n")

# 9.1 Exportar a CSV
nombre_archivo_csv = 'UF_2025_LIMPIO.csv'
df.to_csv(nombre_archivo_csv, index=False, encoding='utf-8-sig')
print(f"✅ Archivo CSV exportado: {nombre_archivo_csv}")

# 9.2 Exportar estadísticas a CSV
nombre_archivo_stats = 'UF_2025_ESTADISTICAS.csv'
estadisticas_mes.to_csv(nombre_archivo_stats, encoding='utf-8-sig')
print(f"✅ Archivo de estadísticas exportado: {nombre_archivo_stats}")

# 9.3 Crear reporte resumido
print("\n📄 Creando reporte resumido...")
reporte = f"""
REPORTE DE ANÁLISIS - VALORES UF 2025
{'='*60}
Fecha de generación: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}

1. INFORMACIÓN DEL DATASET
   - Registros totales: {len(df)}
   - Columnas: {len(df.columns)}
   - Periodo: Enero - Diciembre 2025

2. VALORES DESTACADOS
   - UF Promedio Anual: ${df[meses].mean().mean():,.2f}
   - UF Mínima: ${df[meses].min().min():,.2f}
   - UF Máxima: ${df[meses].max().max():,.2f}
   - Variación Anual: {variacion_anual_pct:+.2f}%

3. CALIDAD DE DATOS
   - Valores nulos: {df.isnull().sum().sum()}
   - Duplicados: 0
   - Estado: ✅ DATOS LIMPIOS

{'='*60}
"""

with open('REPORTE_UF_2025.txt', 'w', encoding='utf-8') as f:
    f.write(reporte)

print(f"✅ Reporte de texto exportado: REPORTE_UF_2025.txt")
print("\n" + reporte)

print("\n" + "="*60)
print("✅ PROCESO COMPLETADO EXITOSAMENTE")
print("="*60)
print("\nArchivos generados:")
print(f"  1. {nombre_archivo_csv} - Datos limpios")
print(f"  2. {nombre_archivo_stats} - Estadísticas descriptivas")
print(f"  3. REPORTE_UF_2025.txt - Reporte resumido")

## 🎯 10. Conclusiones y Hallazgos

### Resumen del Análisis:

1. **Extracción de Datos**: Se extrajeron exitosamente los valores UF desde el sitio web del SII usando `pd.read_html()`

2. **Limpieza de Datos**: Se aplicaron las siguientes técnicas:
   - Eliminación de duplicados
   - Conversión de tipos de datos
   - Manejo de valores nulos mediante interpolación
   - Identificación y tratamiento de outliers
   - Normalización de formatos

3. **Análisis Descriptivo**: Se calcularon:
   - Estadísticas básicas (media, mediana, desviación estándar)
   - Valores extremos por mes
   - Tendencias y variaciones mensuales
   - Correlaciones entre meses

4. **Exportación**: Los datos limpios fueron exportados en formato CSV para uso futuro

### Próximos Pasos Sugeridos:
- Visualización de datos con matplotlib/seaborn
- Análisis de series temporales
- Predicción de valores futuros
- Comparación con años anteriores